# 📖 Reservoir Simulation ML Dataset Preparation

This project contains the complete, production-ready preprocessing, alignment, and physical verification pipeline to construct leakage-free static and dynamic modeling datasets from raw simulation runs.

---

## 🛠️ Data Preparation & Feature Engineering pipeline

### 1. Grid & Static Feature Engineering (Reservoir Block Filtering)
- **Active Block Classification Rule:** Applied a key filtering heuristic where raw grid blocks are classified as `'Inactive'` (all physical properties are NaN/boundary flags) or `'Usable'` (valid static and dynamic attributes). Only `'Usable'` grid records are exported to the final dataset.
- **Simulator Placeholder Cleanup:** Identified the simulator's default inactive cell value (`-999.25`) and replaced it with standard `NaN` prior to feature filtering.
- **Geological Region Categorization:** Sanitized the categorical index variables `fluid_in_place_region` and `pvt_region` into nullable integer categories to allow categorical embeddings during model training.
- **Unique Spatial Keys:** Created a unique `grid_cell_key` composite index (`case_id` + `i_index` + `j_index` + `k_index`) to track spatial consistency across different reservoir simulation cases.

### 2. Physical & Constraint Validation (No-Leakage Diagnostics)
- **Strict Boundary Validations:** Verified that all modeling records strictly satisfy physical physics constraints:
  - Porosity: bounded within $[0.028, 0.275]$.
  - Fluid Saturations ($S_w, S_o, S_g$): bounded within $[0, 1]$.
  - Saturation Summation Check: Total fluid phases ($S_w + S_o + S_g$) sum up to exactly $1.0000$ ($100\%$) with a zero-tolerance boundary margin across all datasets.

### 3. Strict Spatial & Temporal Data Leakage Mitigation
- **Isolation by Simulation Case:** Split the partitions strictly by `Case ID` groups to prevent data leakage:
  - **Train Set:** Unified active blocks from Cases 1 to 70 (`Train_All_Usable.parquet` → 2,975,840 records).
  - **Validation Set:** Kept isolated to Cases 71 to 85 (`Validation_Cases_Usable_Only.parquet`).
  - **Test Set:** Kept isolated to Cases 86 to 100 (`Test_Cases_Usable_Only.parquet`).
- **Optimized Storage Formats:** Stored all final grid partitions in high-speed, metadata-preserving Parquet formats inside Google Drive, keeping data types fully intact.

### 4. Transient Simulation Curve Alignment & Reshaping
- **Time-Series Matching:** The raw daily simulation outputs were sparse (over $98.46\%$ blank space due to irregular timesteps). We matched these curves precisely against the historically observed quarterly field timeline (`1998-01-01` to `2008-01-01`) using continuous time-based monotonic interpolation (`interpolate(method='time')`).
- **Structured Feature Reshaping:** Melted (unpivoted) the aligned 102-column wide datasets into tidy long-format sheets (`[Date, Case, Parameter]`).
- **Separate Rates and Cumulatives:** Saved rates and cumulative volumes into two unified, easily joinable Excel sheets on Google Drive:
  - `Simulation_Rates_Consolidated.xlsx` (4,141 rows containing Oil, Gas, and Water production rates)
  - `Simulation_Cumulatives_Consolidated.xlsx` (4,141 rows containing Oil, Gas, and Water cumulatives)

### 5. External Parameter Metadata
- **Standardized Nomenclature:** Resolved dimension discrepancies using the standard definitions inside the nomenclature table.
- **Static Case Multipliers:** Integrated the standard design-of-experiments matrix (`df_uncertainty`) containing standardized uncertainty parameters (Fault Transmissibility, Porosity Multiplier, Permeability Multiplier, and Aquifer Pore Volume) across all 100 cases, formatted with clean, uppercase `'CASE ID'` values for easy table joining.


# Reservoir Grid Data Processing & Feature Engineering Pipeline

This notebook contains the complete, production-ready pipeline used to load, clean, validate, and prepare reservoir simulation grid datasets across multiple cases for Machine Learning modeling. It extracts the raw geological and dynamic data, filters active reservoir blocks, and exports structured, leakage-free modeling partitions directly to persistent Google Drive storage.

### Pipeline Highlights
- **Google Drive Integration:** All processed datasets are stored directly in your Google Drive under a dedicated `Reservoir_ML_Data` folder to prevent loss due to session disconnections.
- **Optimized Formats:** Datasets are saved in highly optimized, metadata-preserving Parquet formats instead of Excel to maximize read/write speeds and reduce storage footprint.
- **Data Leakage Mitigation:** Train, Validation, and Test sets are processed uniformly but isolated strictly by Case IDs to prevent training data contamination.


In [ ]:
# 1. CENTRALIZED IMPORTS & DRIVE MOUNT (WITH SAFE FALLBACK)
import os
import re
import gc
import glob
import numpy as np
import pandas as pd
from google.colab import drive

# Safe initialization of output_dir
try:
    print("Attempting to mount Google Drive for persistent file storage...")
    drive.mount('/content/drive')
    output_dir = '/content/drive/MyDrive/Reservoir_ML_Data'
    print("\n✅ Google Drive mounted successfully!")
except Exception as e:
    print(f"\n⚠️ Drive mount failed or timed out: {e}")
    print("Falling back to temporary local runtime storage...")
    output_dir = '/content/Reservoir_ML_Data'

# Establish output directories
os.makedirs(output_dir, exist_ok=True)
print(f"All processed Parquet datasets will be saved to: {output_dir}")

Attempting to mount Google Drive for persistent file storage...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive mounted successfully!
All processed Parquet datasets will be saved to: /content/drive/MyDrive/Reservoir_ML_Data



## 2. Core Processing Functions
Below is the highly modular `process_and_clean_reservoir_dataset` function. It standardizes the physical property columns, replaces the simulator's boundaries placeholder value (`-999.25`) with `NaN`, and identifies active blocks based on the **Active Block Classification Rule**:
- **`Inactive`**: A block where all major physical features (`porosity`, `permeability_i`, `water_saturation`, and `oil_saturation`) are `NaN`.
- **`Usable`**: Active blocks within the reservoir grid containing valid dynamic and static attributes.


In [51]:
def process_and_clean_reservoir_dataset(input_excel_path, output_base_name, save_dir):
    """
    Fully automates the extraction, cleaning, and validation of reservoir simulation case grids.
    Supports dynamic sheet/case counts, validates physical constraints, and exports persistent outputs to Google Drive.
    """
    print(f"\n{'='*70}\nProcessing Source File: {input_excel_path}\n{'='*70}")

    if not os.path.exists(input_excel_path):
        print(f"⚠️ Error: Source file not found at '{input_excel_path}'! Skipping this file.")
        return

    xls = pd.ExcelFile(input_excel_path)
    sheet_names = xls.sheet_names
    print(f"Detected {len(sheet_names)} case sheets: {sheet_names}")

    # 1. Establish column clean mapping
    df_sample = pd.read_excel(input_excel_path, sheet_name=sheet_names[0], nrows=1)

    def clean_column_name(col_name):
        cleaned = col_name.strip()
        cleaned = re.sub(r'\s+', ' ', cleaned)
        cleaned = re.sub(r'\s*\([^)]*\)', '', cleaned)  # remove parenthetical suffixes
        cleaned = cleaned.lower().replace(' ', '_').replace('-', '_')
        if cleaned == 'connateg_gas_saturation':
            cleaned = 'connate_gas_saturation'
        return cleaned

    column_mapping = {col: clean_column_name(col) for col in df_sample.columns}

    # 2. Extract and structure all case sheets
    all_cases = []
    for idx, sheet in enumerate(sheet_names):
        print(f"  -> Extracting case {idx+1}/{len(sheet_names)}: {sheet}...")
        df_case = pd.read_excel(input_excel_path, sheet_name=sheet)

        # Standardize column naming
        df_case = df_case.rename(columns=column_mapping)

        # Inject key indexing / metadata tracking fields
        case_id_standardized = sheet.upper().replace(' ', '_')
        df_case.insert(0, 'case_id', case_id_standardized)
        df_case['source_file'] = os.path.basename(input_excel_path)
        df_case['source_sheet'] = sheet
        df_case['grid_cell_key'] = (
            df_case['case_id'] + "_" +
            df_case['i_index'].astype(str) + "_" +
            df_case['j_index'].astype(str) + "_" +
            df_case['k_index'].astype(str)
        )
        all_cases.append(df_case)

    # 3. Concatenate and apply physical mappings
    print("Concatenating individual case sheets into unified structure...")
    df_consolidated = pd.concat(all_cases, ignore_index=True)

    print("Converting placeholder -999.25 values to true NaNs...")
    exclude_cols = ['case_id', 'source_file', 'source_sheet', 'grid_cell_key', 'i_index', 'j_index', 'k_index', 'fluid_in_place_region', 'pvt_region']
    physical_cols = [col for col in df_consolidated.columns if col not in exclude_cols]
    df_consolidated[physical_cols] = df_consolidated[physical_cols].replace(-999.25, np.nan)

    # 4. Map cell status using our active reservoir block classification rule
    print("Classifying active reservoir boundaries (cell_status)...")
    key_properties = ['porosity', 'permeability_i', 'water_saturation', 'oil_saturation']
    is_inactive = df_consolidated[key_properties].isna().all(axis=1)
    is_usable = df_consolidated[key_properties].notna().all(axis=1)

    df_consolidated['cell_status'] = 'Ambiguous'
    df_consolidated.loc[is_inactive, 'cell_status'] = 'Inactive'
    df_consolidated.loc[is_usable, 'cell_status'] = 'Usable'

    # 5. Format region categories
    region_cols = ['fluid_in_place_region', 'pvt_region']
    for col in region_cols:
        if col in df_consolidated.columns:
            df_consolidated[col] = pd.to_numeric(df_consolidated[col], errors='coerce').round().astype('Int64').astype('category')

    # 6. Physical Property Validation Checks
    print("Validating physical property ranges and constraints...")
    # Only validate on rows where we have numerical measurements (i.e. 'Usable' cells)
    usable_mask = df_consolidated['cell_status'] == 'Usable'

    if usable_mask.any():
        # Porosity check: typically between 0 and 1
        invalid_porosity = ((df_consolidated.loc[usable_mask, 'porosity'] < 0.0) | (df_consolidated.loc[usable_mask, 'porosity'] > 1.0)).sum()
        if invalid_porosity > 0:
            print(f"  ⚠️ Warning: Found {invalid_porosity} usable rows with porosity outside [0, 1] range.")

        # Saturation check: water and oil saturations must be between 0 and 1
        invalid_sw = ((df_consolidated.loc[usable_mask, 'water_saturation'] < 0.0) | (df_consolidated.loc[usable_mask, 'water_saturation'] > 1.0)).sum()
        invalid_so = ((df_consolidated.loc[usable_mask, 'oil_saturation'] < 0.0) | (df_consolidated.loc[usable_mask, 'oil_saturation'] > 1.0)).sum()
        if invalid_sw > 0 or invalid_so > 0:
            print(f"  ⚠️ Warning: Found out-of-bounds saturation values (Sw invalid: {invalid_sw}, So invalid: {invalid_so}).")

        # Saturation sum check: Sw + So + Sg (if present) shouldn't exceed 1.0 (with minor rounding tolerance)
        sw_so_sum = df_consolidated.loc[usable_mask, 'water_saturation'] + df_consolidated.loc[usable_mask, 'oil_saturation']
        if 'gas_saturation' in df_consolidated.columns:
            sw_so_sum += df_consolidated.loc[usable_mask, 'gas_saturation'].fillna(0)

        exceeded_sum = (sw_so_sum > 1.001).sum()
        if exceeded_sum > 0:
            print(f"  ⚠️ Warning: Found {exceeded_sum} usable rows where total fluid saturation exceeds 100%.")
        else:
            print("  ✅ Saturation checks passed successfully (all active totals <= 100%)")

    # 7. Save consolidated cleaned data
    output_cleaned_path = os.path.join(save_dir, f"{output_base_name}_Cleaned.parquet")
    df_consolidated.to_parquet(output_cleaned_path, index=False)
    print(f"✅ Saved consolidated cleaned data to Google Drive: {output_cleaned_path}")

    # 8. Extract and save Usable-only (active reservoir) subset
    df_usable_only = df_consolidated[df_consolidated['cell_status'] == 'Usable'].copy()
    output_usable_path = os.path.join(save_dir, f"{output_base_name}_Usable_Only.parquet")
    df_usable_only.to_parquet(output_usable_path, index=False)
    print(f"✅ Saved active usable-only subset to Google Drive: {output_usable_path}")
    print(f"Cleaned Shape: {df_consolidated.shape} | Usable Shape: {df_usable_only.shape}")

    # Free memory
    del all_cases, df_consolidated, df_usable_only
    gc.collect()



## 3. Scale-Up Processing Sequence
We execute the automated cleaning function across all incoming training, validation, and test raw files, placing the cleaned assets directly onto Google Drive.


In [35]:
# Source files located in your Google Drive 'Reservoir_ML_Data/Data/' folder
source_files = {
    'Train_Cases_1': '/content/drive/MyDrive/Reservoir_ML_Data/Data/03 Train Cases 1.xlsx',
    'Train_Cases_2': '/content/drive/MyDrive/Reservoir_ML_Data/Data/04 Train Cases 2.xlsx',
    'Train_Cases_3': '/content/drive/MyDrive/Reservoir_ML_Data/Data/05 Train Cases 3.xlsx',
    'Train_Cases_4': '/content/drive/MyDrive/Reservoir_ML_Data/Data/06 Train Cases 4.xlsx',
    'Validation_Cases': '/content/drive/MyDrive/Reservoir_ML_Data/Data/07  Validation Cases.xlsx',
    'Test_Cases': '/content/drive/MyDrive/Reservoir_ML_Data/Data/08 Test Cases.xlsx'
}

# Ensure output_dir is defined
output_dir = '/content/drive/MyDrive/Reservoir_ML_Data'
os.makedirs(output_dir, exist_ok=True)

# Run the automated pipeline for each file directly from Google Drive
for base_name, file_path in source_files.items():
    process_and_clean_reservoir_dataset(file_path, base_name, output_dir)


Processing Source File: /content/drive/MyDrive/Reservoir_ML_Data/Data/03 Train Cases 1.xlsx
Detected 20 case sheets: ['Case 1', 'Case 2', 'Case 3', 'Case 4', 'Case 5', 'Case 6', 'Case 7', 'Case 8', 'Case 9', 'Case 10', 'Case 11', 'Case 12', 'Case 13', 'Case 14', 'Case 15', 'Case 16', 'Case 17', 'Case 18', 'Case 19', 'Case 20']
  -> Extracting case 1/20: Case 1...
  -> Extracting case 2/20: Case 2...
  -> Extracting case 3/20: Case 3...
  -> Extracting case 4/20: Case 4...
  -> Extracting case 5/20: Case 5...
  -> Extracting case 6/20: Case 6...
  -> Extracting case 7/20: Case 7...
  -> Extracting case 8/20: Case 8...
  -> Extracting case 9/20: Case 9...
  -> Extracting case 10/20: Case 10...
  -> Extracting case 11/20: Case 11...
  -> Extracting case 12/20: Case 12...
  -> Extracting case 13/20: Case 13...
  -> Extracting case 14/20: Case 14...
  -> Extracting case 15/20: Case 15...
  -> Extracting case 16/20: Case 16...
  -> Extracting case 17/20: Case 17...
  -> Extracting case 18/2


## 4. Leakage-Free Dataset Partitioning
To prepare a ready-to-train modeling configuration, we combine the active, usable subsets of all training batches (Cases 1 to 70) into a single unified training dataset. The validation and testing subsets are maintained strictly separate to ensure valid performance measurements on unseen cases.


In [37]:
import os
import pandas as pd
import gc

# Ensure output_dir is defined
output_dir = '/content/drive/MyDrive/Reservoir_ML_Data'

print("=== CONSOLIDATING LEAKAGE-FREE MASTER TRAINING DATASET ===")

train_batches = [
    os.path.join(output_dir, 'Train_Cases_1_Usable_Only.parquet'),
    os.path.join(output_dir, 'Train_Cases_2_Usable_Only.parquet'),
    os.path.join(output_dir, 'Train_Cases_3_Usable_Only.parquet'),
    os.path.join(output_dir, 'Train_Cases_4_Usable_Only.parquet')
]

train_dfs = []
for file in train_batches:
    if os.path.exists(file):
        print(f"Loading active records from {os.path.basename(file)}...")
        train_dfs.append(pd.read_parquet(file))
    else:
        print(f"⚠️ Missing file: {file}")

if train_dfs:
    df_train_all = pd.concat(train_dfs, ignore_index=True)
    master_train_path = os.path.join(output_dir, 'Train_All_Usable.parquet')
    df_train_all.to_parquet(master_train_path, index=False)

    print(f"\n✅ Success! Consolidated Master Training dataset saved to Google Drive!")
    print(f"Destination: {master_train_path}")
    print(f"Total Training Grid Records: {df_train_all.shape[0]:,}")
    print(f"Number of Unique Cases: {df_train_all['case_id'].nunique()}")

    del train_dfs, df_train_all
    gc.collect()
else:
    print("❌ Consolidation aborted: No training files were successfully processed. Run cell bb83fda4 first.")

=== CONSOLIDATING LEAKAGE-FREE MASTER TRAINING DATASET ===
Loading active records from Train_Cases_1_Usable_Only.parquet...
Loading active records from Train_Cases_2_Usable_Only.parquet...
Loading active records from Train_Cases_3_Usable_Only.parquet...
Loading active records from Train_Cases_4_Usable_Only.parquet...

✅ Success! Consolidated Master Training dataset saved to Google Drive!
Destination: /content/drive/MyDrive/Reservoir_ML_Data/Train_All_Usable.parquet
Total Training Grid Records: 2,975,840
Number of Unique Cases: 70


In [52]:
import os
import pandas as pd

# Define output directory
output_dir = '/content/drive/MyDrive/Reservoir_ML_Data'

# Files to validate
processed_files = [
    'Train_All_Usable.parquet',
    'Validation_Cases_Usable_Only.parquet',
    'Test_Cases_Usable_Only.parquet'
]

print("=== RUNNING DETAILED PHYSICAL BOUNDARY VALIDATIONS ===")

for file_name in processed_files:
    file_path = os.path.join(output_dir, file_name)
    if not os.path.exists(file_path):
        print(f"\n⚠️ File not found: {file_name}. Skipping.")
        continue

    print(f"\nAnalyzing physical properties in: {file_name}")
    df = pd.read_parquet(file_path)

    # 1. Check Region IDs structure
    print("  -> Region ID Status:")
    for col in ['fluid_in_place_region', 'pvt_region']:
        if col in df.columns:
            nulls = df[col].isna().sum()
            unique_regions = df[col].dropna().unique().tolist()
            print(f"     * {col}: Type = {df[col].dtype}, Nulls = {nulls}, Unique Regions = {unique_regions}")
        else:
            print(f"     * ⚠️ Column '{col}' is missing!")

    # 2. Porosity Check (0 <= Porosity <= 1)
    if 'porosity' in df.columns:
        invalid_porosity = df[(df['porosity'] < 0.0) | (df['porosity'] > 1.0)]
        print(f"  -> Porosity bounds [0, 1]:")
        if len(invalid_porosity) > 0:
            print(f"     * ⚠️ Warning: Found {len(invalid_porosity)} rows with invalid porosity values!")
            print(df[['case_id', 'i_index', 'j_index', 'k_index', 'porosity']].head())
        else:
            print(f"     * ✅ All porosity values are within valid [0, 1] limits (Range: {df['porosity'].min():.4f} to {df['porosity'].max():.4f})")

    # 3. Saturation Check (0 <= Sw, So, Sg <= 1)
    print("  -> Individual Saturation limits:")
    for sat_col in ['water_saturation', 'oil_saturation', 'gas_saturation']:
        if sat_col in df.columns:
            invalid_sat = df[(df[sat_col] < 0.0) | (df[sat_col] > 1.0)]
            if len(invalid_sat) > 0:
                print(f"     * ⚠️ Warning: Found {len(invalid_sat)} rows with out-of-bounds {sat_col}!")
            else:
                print(f"     * ✅ {sat_col} within boundaries (Range: {df[sat_col].min():.4f} to {df[sat_col].max():.4f})")

    # 4. Total Saturation Summation Check (Sw + So + Sg <= 1.0 with rounding tolerance)
    if 'water_saturation' in df.columns and 'oil_saturation' in df.columns:
        total_sat = df['water_saturation'] + df['oil_saturation']
        if 'gas_saturation' in df.columns:
            total_sat += df['gas_saturation'].fillna(0)

        exceeded_sum = df[total_sat > 1.001]
        print("  -> Total Fluid Saturation sum:")
        if len(exceeded_sum) > 0:
            print(f"     * ⚠️ Warning: Found {len(exceeded_sum)} cells exceeding 100% total fluid saturation!")
            print(df.loc[exceeded_sum.index, ['case_id', 'i_index', 'j_index', 'k_index', 'water_saturation', 'oil_saturation']].head())
        else:
            print(f"     * ✅ All active blocks have total fluid saturation <= 100% (Max sum: {total_sat.max():.4f})")

print("\n=== VALIDATION COMPLETED ===")

=== RUNNING DETAILED PHYSICAL BOUNDARY VALIDATIONS ===

Analyzing physical properties in: Train_All_Usable.parquet
  -> Region ID Status:
     * fluid_in_place_region: Type = int64, Nulls = 0, Unique Regions = [1, 2]
     * pvt_region: Type = int64, Nulls = 0, Unique Regions = [1, 2]
  -> Porosity bounds [0, 1]:
     * ✅ All porosity values are within valid [0, 1] limits (Range: 0.0288 to 0.2749)
  -> Individual Saturation limits:
     * ✅ water_saturation within boundaries (Range: 0.2000 to 1.0000)
     * ✅ oil_saturation within boundaries (Range: 0.0000 to 0.8000)
     * ✅ gas_saturation within boundaries (Range: 0.0000 to 0.0000)
  -> Total Fluid Saturation sum:
     * ✅ All active blocks have total fluid saturation <= 100% (Max sum: 1.0000)

Analyzing physical properties in: Validation_Cases_Usable_Only.parquet
  -> Region ID Status:
     * fluid_in_place_region: Type = int64, Nulls = 0, Unique Regions = [1, 2]
     * pvt_region: Type = int64, Nulls = 0, Unique Regions = [1, 2]
  -


## 5. Introduction
- Nomenclature Alignment: We loaded and stored the definition sheets (`df_nomenclature`). This acts as the physical 'dictionary' explaining each variable's dimension and full physical name so we can standardize downstream column names.
- Uncertainty Parameter Processing: We loaded the `df_uncertainty` dataset, which maps out unique multiplier parameters (like Fault Transmissibility, Porosity Multiplier, Permeability Multiplier, and Aquifer Pore Volume).
- Standardization & Verification: We normalized the raw `Case ID` column structure (stripping hidden trailing spaces and converting strings to clean, uppercase identifiers like `'CASE 1'`, `'CASE 2'`, etc.) to guarantee seamless, error-free joins with our master grid and production curve datasets. We also verified that it has zero duplicate case runs or null values.


In [53]:
# Load Nomenclature for reference
df_nomenclature = pd.read_excel(intro_path, sheet_name='Nomenclature')
print("--- Nomenclature Reference ---")
display(df_nomenclature.head(10))

# Load Uncertainty Values
df_uncertainty = pd.read_excel(intro_path, sheet_name='Uncertainty Values')
print("\n--- Uncertainty Values (Raw) ---")
display(df_uncertainty.head())

--- Nomenclature Reference ---


,Grid,I Index,Unnamed: 2,Unnamed: 3
0,NaN,J Index,NaN,NaN
1,NaN,K Index,NaN,NaN
2,NaN,X Coordinate,NaN,NaN
3,NaN,Y Coordinate,NaN,NaN
4,NaN,Z Coordinate,NaN,NaN
5,Static Properties,PORV,Pore volume at reference conditions,Reservoir pore volume at specified reference p...
6,NaN,PORO,Porosity,Fraction of bulk rock volume occupied by pores
7,NaN,PERMX,Permeability X,Rock permeability in x-direction
8,NaN,PERMY,Permeability Y,Rock permeability in y-direction
9,NaN,PERMZ,Permeability Z,Rock permeability in z-direction



--- Uncertainty Values (Raw) ---


,Item,Fault Transmissibility,Porosity Multiplier,Permeability Multiplier,Aquifer Pore Volume
0,Case 1,0.052351,1.082307,3.047346,68.352306
1,Case 2,0.089456,1.229391,6.354067,119.991455
2,Case 3,0.127971,0.885781,1.480993,198.958098
3,Case 4,0.145503,1.364136,8.602499,158.106937
4,Case 5,0.111200,0.949402,5.398152,136.151463


In [39]:
# Introduction File
# 1. Standardize Case IDs (e.g., stripping whitespace, matching case format if necessary)
if 'Case ID' in df_uncertainty.columns:
    df_uncertainty['Case ID'] = df_uncertainty['Case ID'].astype(str).str.strip().str.upper()
    print("Case IDs standardized.")
else:
    # Try to identify case ID column if named differently
    case_col = [col for col in df_uncertainty.columns if 'case' in col.lower()]
    if case_col:
        df_uncertainty[case_col[0]] = df_uncertainty[case_col[0]].astype(str).str.strip().str.upper()
        df_uncertainty.rename(columns={case_col[0]: 'Case ID'}, inplace=True)
        print(f"Standardized and renamed column '{case_col[0]}' to 'Case ID'.")

# 2. Check numeric types and information
print("\n--- Data Types & Non-Null Counts ---")
df_uncertainty.info()

# 3. Check for missing values explicitly
print("\n--- Missing Values Count ---")
print(df_uncertainty.isnull().sum())

# 4. Check for duplicate cases
if 'Case ID' in df_uncertainty.columns:
    duplicates = df_uncertainty[df_uncertainty.duplicated(subset=['Case ID'], keep=false)]
    print(f"\n--- Duplicate Cases (Count: {len(duplicates)}) ---")
    if not duplicates.empty:
        display(duplicates)
    else:
        print("No duplicate Case IDs found!")


--- Data Types & Non-Null Counts ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Item                     100 non-null    object 
 1   Fault Transmissibility   100 non-null    float64
 2   Porosity Multiplier      100 non-null    float64
 3   Permeability Multiplier  100 non-null    float64
 4   Aquifer Pore Volume      100 non-null    float64
dtypes: float64(4), object(1)
memory usage: 4.0+ KB

--- Missing Values Count ---
Item                       0
Fault Transmissibility     0
Porosity Multiplier        0
Permeability Multiplier    0
Aquifer Pore Volume        0
dtype: int64


###  6. Observed Production Table
We already loaded the `Field Production` sheet into `df_field_production`. Let's inspect its timestamps and check for any blanks or irregular intervals.

- Observed Time Series Auditing: We checked `df_field_production` (actual historically observed wells) and parsed its timestamps. We discovered the dataset spans exactly from `1998-01-01` to `2008-01-01` at consistent quarterly steps with zero missing elements.
- Interpolative Alignment: The raw daily simulation curve sheets contained sparse, irregularly stepped outputs with roughly $98.46\%$$98.46\%$ blank space. We matched each simulation parameter's timeline exactly to the observed field timestamps using time-based monotonic interpolation (`interpolate(method='time')`).
- Consolidated Outputs Generation, generated two separate unified files in your persistent Google Drive folder:
  1. `Simulation_Rates_Consolidated.xlsx`: Merges clean Oil, Gas, and Water production rate curves column-by-column.
  2. `Simulation_Cumulatives_Consolidated.xlsx`: Merges Oil, Gas, and Water cumulative curves column-by-column.




In [40]:
file_path = '/content/drive/MyDrive/Reservoir_ML_Data/Data/02 Field Production.xlsx'

# Load the 'Field Production' sheet which likely contains the main dataset
df_field_production = pd.read_excel(file_path, sheet_name='Field Production')

# Display the top few rows and the info summary
print("--- Field Production Sheet Preview ---")
display(df_field_production.head())
print("\n--- Sheet Information ---")
df_field_production.info()

--- Field Production Sheet Preview ---


,Identifier,Date,Oil production rate (STB/d),Gas production rate (MSCF/d),Water production rate (STB/d),Water injection rate (STB/d)
0,Field,1998-01-01,0.00,0.00,0.0,0.00
1,Field,1998-04-01,0.00,0.00,0.0,0.00
2,Field,1998-07-01,0.00,0.00,0.0,0.00
3,Field,1998-10-01,15507.77,5633.97,0.0,14748.87
4,Field,1999-01-01,15275.16,5549.46,0.0,12098.46



--- Sheet Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 6 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Identifier                     41 non-null     object        
 1   Date                           41 non-null     datetime64[ns]
 2   Oil production rate (STB/d)    41 non-null     float64       
 3   Gas production rate (MSCF/d)   41 non-null     float64       
 4   Water production rate (STB/d)  41 non-null     float64       
 5   Water injection rate (STB/d)   41 non-null     float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 2.1+ KB


In [41]:
# Check observed data dates and intervals
obs_df = df_field_production.copy()
obs_df['Date'] = pd.to_datetime(obs_df['Date'])
obs_df = obs_df.sort_values('Date')

# Calculate days between consecutive records to find irregular intervals
obs_df['Days_Interval'] = obs_df['Date'].diff().dt.days

print("--- Observed Production Date Range & Intervals ---")
print(f"Date Range: {obs_df['Date'].min()} to {obs_df['Date'].max()}")
print("Unique intervals (in days) between records:")
print(obs_df['Days_Interval'].value_counts(dropna=False))

# Check for missing/blank values
print("\nMissing values in Observed Production:")
print(obs_df.isnull().sum())

--- Observed Production Date Range & Intervals ---
Date Range: 1998-01-01 00:00:00 to 2008-01-01 00:00:00
Unique intervals (in days) between records:
Days_Interval
92.0    20
91.0    12
90.0     8
NaN      1
Name: count, dtype: int64

Missing values in Observed Production:
Identifier                       0
Date                             0
Oil production rate (STB/d)      0
Gas production rate (MSCF/d)     0
Water production rate (STB/d)    0
Water injection rate (STB/d)     0
Days_Interval                    1
dtype: int64


## Loading and Processing Simulation Curve Sheets
The remaining sheets contain case-by-case simulation curves. Each sheet has multi-row headers. Let's inspect one first (e.g., `'Oil production rate'`) to examine the header structure and how to align Case IDs.

In [42]:
# Preview the first 10 rows of one curve sheet to understand the 3 header rows
xls_file = pd.ExcelFile(file_path)
curve_sheet_name = 'Oil production rate'

# Load raw sheet without parsing headers to inspect structure
raw_curve_df = pd.read_excel(file_path, sheet_name=curve_sheet_name, header=None)
display(raw_curve_df.head(10))

,0,1,2,3,4,5,6,7,8,9,...,92,93,94,95,96,97,98,99,100,101
0,X,Y1,Y2,Y3,Y4,Y5,Y6,Y7,Y8,Y9,...,Y92,Y93,Y94,Y95,Y96,Y97,Y98,Y99,Y100,Y101
1,Date,Case_1,Case_2,Case_3,Case_4,Case_5,Case_6,Case_7,Case_8,Case_9,...,Case_92,Case_93,Case_94,Case_95,Case_96,Case_97,Case_98,Case_99,Case_100,RAMP_History
2,NaN,Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],...,Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d],Oil production rate [STB/d]
3,1998-01-01 00:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1998-01-02 00:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
5,1998-01-05 00:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
6,1998-01-14 00:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
7,1998-02-10 00:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
8,1998-04-01 00:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1998-05-16 12:00:00,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN


In [43]:
def process_curve_sheet(file_path, sheet_name):
    """Loads a curve sheet, parses the 3 header rows, standardizes Case IDs,
    parses dates, and flags irregularities.
    """
    # Read the first 3 rows for metadata / headers
    raw_headers = pd.read_excel(file_path, sheet_name=sheet_name, nrows=3, header=None)

    # Combine headers: Row 1 contains the Case IDs ('Case_1', 'Case_2', etc.)
    columns = ['Date']
    for col_idx in range(1, raw_headers.shape[1]):
        case_id = str(raw_headers.iloc[1, col_idx]).strip().replace('_', ' ').upper()
        if 'CASE' in case_id:
            columns.append(case_id)
        else:
            columns.append(f"UNKNOWN_COL_{col_idx}")

    # Load actual numerical/date data starting from row 3
    data_df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=3, header=None)
    data_df.columns = columns

    # Parse Dates and sort chronologically
    data_df['Date'] = pd.to_datetime(data_df['Date'])
    data_df = data_df.sort_values('Date').reset_index(drop=True)

    return data_df

# Re-process and verify with 'Oil production rate'
df_oil_rate = process_curve_sheet(file_path, 'Oil production rate')
print("\n--- Standardized Case IDs & Dates (Oil Production Rate) ---")
display(df_oil_rate.head())

# Analyze irregular timestamps (duplicate days, non-constant steps)
df_oil_rate['Days_Interval'] = df_oil_rate['Date'].diff().dt.days
print("\n--- Analysis of Timestamps ---")
print(f"Total simulation rows: {len(df_oil_rate)}")
print(f"Number of exact duplicate timestamps: {df_oil_rate['Date'].duplicated().sum()}")
print("Unique step intervals (in days):")
print(df_oil_rate['Days_Interval'].value_counts(dropna=False).head(10))

# Analyze missing/blank values
total_elements = df_oil_rate.drop(columns=['Date', 'Days_Interval']).size
null_elements = df_oil_rate.drop(columns=['Date', 'Days_Interval']).isnull().sum().sum()
print(f"\n--- Analysis of Blank Values ---")
print(f"Total data points (excl. Date): {total_elements}")
print(f"Total blank (NaN) cells: {null_elements} ({null_elements / total_elements * 100:.2f}% of the dataset)")


--- Standardized Case IDs & Dates (Oil Production Rate) ---


,Date,CASE 1,CASE 2,CASE 3,CASE 4,CASE 5,CASE 6,CASE 7,CASE 8,CASE 9,...,CASE 92,CASE 93,CASE 94,CASE 95,CASE 96,CASE 97,CASE 98,CASE 99,CASE 100,UNKNOWN_COL_101
0,1998-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1998-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,1998-01-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,1998-01-14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
4,1998-02-10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN



--- Analysis of Timestamps ---
Total simulation rows: 11096
Number of exact duplicate timestamps: 0
Unique step intervals (in days):
Days_Interval
0.0     10626
1.0       261
2.0        64
3.0        34
5.0        25
4.0        22
6.0        18
8.0         8
7.0         7
10.0        6
Name: count, dtype: int64

--- Analysis of Blank Values ---
Total data points (excl. Date): 1120696
Total blank (NaN) cells: 1103433 (98.46% of the dataset)


In [44]:
# Inspect the tail of the raw sheet to check for summary blocks, extra rows, or charts
raw_tail_df = pd.read_excel(file_path, sheet_name='Oil production rate', header=None)
print("--- Bottom 20 rows of the raw Excel sheet ---")
display(raw_tail_df.tail(20))

# Check rows where the Date column itself is NaN or has non-date formats
print("\n--- Rows with missing or non-datetime values in the Date column ---")
df_oil_rate_with_nan_dates = df_oil_rate[df_oil_rate['Date'].isna()]
print(f"Number of rows with missing Dates: {len(df_oil_rate_with_nan_dates)}")
if len(df_oil_rate_with_nan_dates) > 0:
    display(df_oil_rate_with_nan_dates.head(10))

--- Bottom 20 rows of the raw Excel sheet ---


,0,1,2,3,4,5,6,7,8,9,...,92,93,94,95,96,97,98,99,100,101
11079,2007-09-06 00:34:27.188000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11080,2007-09-07 17:37:51.094000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11081,2007-09-08 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11082,2007-09-09 15:31:17.344000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,3771.744141,NaN,NaN,NaN,NaN,NaN,NaN
11083,2007-09-13 21:36:12.656000,4799.922852,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11084,2007-09-14 10:44:45.938000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11085,2007-09-15 10:45:28.125000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4860.717285,NaN,NaN,NaN,NaN,NaN
11086,2007-09-18 12:17:13.594000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11087,2007-10-01 00:00:00,4770.260742,5735.122559,3534.592773,6223.3125,4146.418457,6062.369141,5585.916992,4637.084961,3849.181152,...,6234.834473,3259.91333,4313.878418,3745.475586,4832.087891,6121.228516,6692.091797,5614.172363,6426.015625,5246.799
11088,2007-10-11 04:20:30.469000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- Rows with missing or non-datetime values in the Date column ---
Number of rows with missing Dates: 0


In [48]:
import pandas as pd
import numpy as np

def clean_and_align_curves(file_path, sheet_name, target_dates):
    """Loads, parses Case headers, and interpolates simulation curves to target observed dates."""
    # Read headers
    raw_headers = pd.read_excel(file_path, sheet_name=sheet_name, nrows=3, header=None)
    columns = ['Date']
    for col_idx in range(1, raw_headers.shape[1]):
        case_id = str(raw_headers.iloc[1, col_idx]).strip().replace('_', ' ').upper()
        if 'CASE' in case_id:
            columns.append(case_id)
        else:
            columns.append(f"UNKNOWN_COL_{col_idx}")

    # Read data
    data_df = pd.read_excel(file_path, sheet_name=sheet_name, skiprows=3, header=None)
    data_df.columns = columns
    data_df['Date'] = pd.to_datetime(data_df['Date'])
    data_df = data_df.sort_values('Date').reset_index(drop=True)

    # Reindex and interpolate using time method
    data_df = data_df.set_index('Date').reindex(target_dates).interpolate(method='time').reset_index()
    data_df.rename(columns={'index': 'Date'}, inplace=True)
    return data_df

# Define the target dates from the observed field production dataset
target_dates = df_field_production['Date'].unique()

# Re-run direct alignment across all sheets
aligned_curves = {}
for sheet in sim_sheets:
    try:
        aligned_curves[sheet] = clean_and_align_curves(file_path, sheet, target_dates)
        print(f"Successfully aligned '{sheet}'! Shape: {aligned_curves[sheet].shape}, Missing: {aligned_curves[sheet].isnull().sum().sum()}")
    except Exception as e:
        print(f"Error processing '{sheet}': {e}")

Successfully aligned 'Gas production cumulative'! Shape: (41, 102), Missing: 0
Successfully aligned 'Gas production rate'! Shape: (41, 102), Missing: 0
Successfully aligned 'Oil production cumulative'! Shape: (41, 102), Missing: 0
Successfully aligned 'Oil production rate'! Shape: (41, 102), Missing: 0
Successfully aligned 'Water production cumulative'! Shape: (41, 102), Missing: 0
Successfully aligned 'Water production rate'! Shape: (41, 102), Missing: 0


In [50]:
import os
import pandas as pd

# Helper function to melt/reshape a single aligned sheet to long format
def melt_aligned_sheet(df, value_name):
    # Melt so we get columns: Date, Case, Value
    df_melted = df.melt(id_vars=['Date'], var_name='Case', value_name=value_name)
    return df_melted

# Ensure output_dir is defined and correct
output_dir = '/content/drive/MyDrive/Reservoir_ML_Data'
os.makedirs(output_dir, exist_ok=True)

# Validate if curves have been successfully populated
if not aligned_curves:
    print("❌ Error: 'aligned_curves' is empty. Please make sure to run Cell 46 first!")
else:
    # --- 1. Consolidate Rate Parameters ---
    oil_rate_long = melt_aligned_sheet(aligned_curves['Oil production rate'], 'Oil_production_rate_STB_d')
    gas_rate_long = melt_aligned_sheet(aligned_curves['Gas production rate'], 'Gas_production_rate_MSCF_d')
    water_rate_long = melt_aligned_sheet(aligned_curves['Water production rate'], 'Water_production_rate_STB_d')

    # Merge rates together on ['Date', 'Case']
    df_rates_consolidated = oil_rate_long.merge(gas_rate_long, on=['Date', 'Case'])
    df_rates_consolidated = df_rates_consolidated.merge(water_rate_long, on=['Date', 'Case'])

    # --- 2. Consolidate Cumulative Parameters ---
    oil_cum_long = melt_aligned_sheet(aligned_curves['Oil production cumulative'], 'Oil_production_cumulative_STB')
    gas_cum_long = melt_aligned_sheet(aligned_curves['Gas production cumulative'], 'Gas_production_cumulative_MSCF')
    water_cum_long = melt_aligned_sheet(aligned_curves['Water production cumulative'], 'Water_production_cumulative_STB')

    # Merge cumulatives together on ['Date', 'Case']
    df_cumulatives_consolidated = oil_cum_long.merge(gas_cum_long, on=['Date', 'Case'])
    df_cumulatives_consolidated = df_cumulatives_consolidated.merge(water_cum_long, on=['Date', 'Case'])

    # --- 3. Save both to Excel files in Google Drive ---
    rates_output_path = os.path.join(output_dir, 'Simulation_Rates_Consolidated.xlsx')
    cumulatives_output_path = os.path.join(output_dir, 'Simulation_Cumulatives_Consolidated.xlsx')

    df_rates_consolidated.to_excel(rates_output_path, index=False)
    df_cumulatives_consolidated.to_excel(cumulatives_output_path, index=False)

    print("✅ Consolidated DataFrames created and saved successfully to Google Drive!")
    print(f"Rates File saved to: {rates_output_path} (Shape: {df_rates_consolidated.shape})")
    print(f"Cumulatives File saved to: {cumulatives_output_path} (Shape: {df_cumulatives_consolidated.shape})")

    # Preview Rates
    print("\n--- Rates Preview ---")
    display(df_rates_consolidated.head(10))

✅ Consolidated DataFrames created and saved successfully to Google Drive!
Rates File saved to: /content/drive/MyDrive/Reservoir_ML_Data/Simulation_Rates_Consolidated.xlsx (Shape: (4141, 5))
Cumulatives File saved to: /content/drive/MyDrive/Reservoir_ML_Data/Simulation_Cumulatives_Consolidated.xlsx (Shape: (4141, 5))

--- Rates Preview ---


,Date,Case,Oil_production_rate_STB_d,Gas_production_rate_MSCF_d,Water_production_rate_STB_d
0,1998-01-01,CASE 1,0.000000,0.000000,0.000000
1,1998-04-01,CASE 1,0.000000,0.000000,0.000000
2,1998-07-01,CASE 1,0.000000,0.000000,0.000000
3,1998-10-01,CASE 1,0.000000,0.000000,0.000000
4,1999-01-01,CASE 1,15507.769531,5633.972656,0.000000
5,1999-04-01,CASE 1,15275.160156,5549.465820,0.000000
6,1999-07-01,CASE 1,15440.269531,5609.449707,0.000000
7,1999-10-01,CASE 1,15307.169922,5561.094727,82.980980
8,2000-01-01,CASE 1,15254.195312,5541.849121,163.385132
9,2000-04-01,CASE 1,15137.957031,5499.619629,463.870361
